# Quantum phase estimation on noisy hardware

| | |
|---|---|
| **Level** | Intermediate to advanced |
| **Time** | 60 to 90 minutes |
| **Prerequisites** | The quantum Fourier transform and phase estimation. The zero-noise extrapolation sections are optional. |
| **Default devices** | Rigetti Cepheus |
| **Hardware jobs** | 7 |
| **Approximate cost** | Rigetti is billed by execution time, about 10 credits per job in our tests |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*Part of the QUEST notebooks from qBraid: algorithms.*

Quantum phase estimation (QPE) finds the eigenvalues of a unitary operator. It is the core of Shor's algorithm and of many quantum chemistry methods. In textbooks, adding precision qubits always gives a more accurate answer. This notebook tests that on real hardware.

We apply QPE to an operator whose eigenvalue we know exactly, and vary the number of precision qubits from 2 to 5:

1. On the ideal simulator, the error shrinks with every added qubit, as the theory predicts.
2. On hardware, each added qubit also makes the circuit longer, so noise grows. Past some point, more precision qubits give a worse answer.

The optional last part introduces zero-noise extrapolation, an error-mitigation method that recovers some of the lost accuracy at the cost of more shots.

**Learning objectives**

1. Build QPE for a known test operator.
2. Relate the number of precision qubits to the expected accuracy.
3. Measure how noise limits the useful depth of QPE on a real device.
4. (Optional) Apply zero-noise extrapolation.
5. Judge when QPE is the right tool and when a variational method is more practical.


## QPE in brief

Suppose $U|\psi\rangle = e^{2\pi i \phi}|\psi\rangle$. QPE estimates $\phi \in [0, 1)$ using $t$ precision qubits (also called counting qubits), and returns $\phi$ to $t$ binary digits.

The circuit has four steps:

1. Put the precision qubits into an equal superposition with Hadamard gates.
2. For each precision qubit $j$, apply controlled-$U^{2^j}$ to the target.
3. Apply the inverse quantum Fourier transform to the precision qubits.
4. Measure them.

Each added precision qubit halves the error, but it also doubles the number of controlled-$U$ applications in the deepest part of the circuit. On current hardware, that depth is usually the limit.

Our test operator is the phase gate $U = P(\theta)$ on one target qubit prepared in $|1\rangle$:

$$P(\theta)|1\rangle = e^{i\theta}|1\rangle, \qquad \phi = \theta / 2\pi.$$

We use $\phi = 5/16 = 0.0101_2$. It has an exact 4-bit binary form, so with $t = 4$ the ideal simulator should return it with no error.

## Setup

In [ ]:
# Standard scientific Python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# qBraid
from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

# The exact phase we're trying to estimate
TRUE_PHI = 5 / 16   # = 0.3125
TRUE_THETA = 2 * np.pi * TRUE_PHI  # = 5*pi/8

print(f"True phase: phi = {TRUE_PHI} (exact 4-bit binary: 0.0101)")
print(f"True angle: theta = {TRUE_THETA:.6f} rad")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
SHOTS_HW = 500
SHOTS_ZNE = 500
QUEST_JOB_TAGS = {"quest": "algo-qpenoise"}   # labels this notebook's hardware jobs for QUEST usage statistics


## Building QPE

The function below builds QPE with $t$ precision qubits. The target is a single qubit prepared in $|1\rangle$, which is an eigenstate of the phase gate.

In [ ]:
def build_qpe_circuit(t, theta=TRUE_THETA):
    """
    Build a QPE circuit with `t` precision qubits.
    Target register is 1 qubit prepared in |1> (an eigenstate of P(theta)).
    Estimates phi where P(theta)|1> = e^(i * 2*pi*phi) |1>.
    """
    precision = QuantumRegister(t, 'p')
    target = QuantumRegister(1, 'psi')
    creg = ClassicalRegister(t, 'c')
    qc = QuantumCircuit(precision, target, creg)

    # Prepare target in |1> (eigenstate)
    qc.x(target[0])

    # Uniform superposition on precision qubits
    qc.h(precision)

    # Controlled-U^(2^j) applications
    # For P(theta), U^(2^j) = P(2^j * theta)
    for j in range(t):
        power = 2 ** j
        qc.cp(power * theta, precision[j], target[0])

    # Inverse QFT on precision register
    qc.append(QFT(t, inverse=True, do_swaps=True).to_gate(), precision)

    # Measure precision register
    qc.measure(precision, creg)
    return qc


# Sanity-check the circuit at t=4
qc_test = build_qpe_circuit(t=4)
print(f"t=4 circuit depth: {qc_test.depth()}")
print(f"t=4 gate count: {sum(qc_test.count_ops().values())}")
qc_test.draw('mpl', fold=100)

## From measurements to a phase estimate

This helper turns a measured bitstring into a phase estimate. The last qubit of the precision register gives the most significant bit.

In [ ]:
def bitstring_to_phase(bitstring, t):
    """
    Convert a QPE measurement bitstring to a phase estimate.
    Qiskit returns bitstrings in little-endian order (leftmost char is highest-indexed qubit).
    """
    # Reverse to little-endian, then interpret as fractional binary
    integer = int(bitstring, 2)
    return integer / (2 ** t)


def counts_to_phase_estimate(counts, t):
    """
    Given a dict of {bitstring: count}, return the maximum-likelihood phase estimate.
    """
    total = sum(counts.values())
    weighted = 0.0
    # Use the most probable bitstring as the estimate
    best_bitstring = max(counts, key=counts.get)
    return bitstring_to_phase(best_bitstring, t)


def counts_to_expected_phase(counts, t):
    """
    Return the expected value of phase across the measurement distribution,
    accounting for the circular nature of phase.
    """
    total = sum(counts.values())
    # Compute average of complex exponentials, then extract angle
    z_sum = 0j
    for bitstring, count in counts.items():
        phase = bitstring_to_phase(bitstring, t)
        z_sum += (count / total) * np.exp(2j * np.pi * phase)
    return (np.angle(z_sum) / (2 * np.pi)) % 1.0


# Verify on ideal simulator
sim = AerSimulator()
result = sim.run(transpile(qc_test, sim), shots=4096).result()
counts = result.get_counts()

est_ml = counts_to_phase_estimate(counts, t=4)
est_avg = counts_to_expected_phase(counts, t=4)
print(f"Max-likelihood estimate at t=4: {est_ml} (true: {TRUE_PHI})")
print(f"Circular-mean estimate at t=4: {est_avg:.6f} (true: {TRUE_PHI})")
print(f"Bitstring '0101' probability: {counts.get('0101', 0) / 4096:.3f}")

With $t = 4$, the phase $5/16$ is exact in 4 binary digits, so the ideal simulator should return `0101` on every shot. Any other result means the circuit is wrong.

## Precision sweep on the ideal simulator

We now vary $t$ from 2 to 5 and record the error of each estimate.

In [ ]:
def error_vs_true(estimate, true=TRUE_PHI):
    """Return absolute error on the circle (handles wraparound)."""
    d = abs(estimate - true)
    return min(d, 1 - d)  # circular distance


t_range = list(range(2, 6))
SHOTS_SIM = 4096

ideal_errors = []
for t in t_range:
    qc = build_qpe_circuit(t=t)
    result = sim.run(transpile(qc, sim), shots=SHOTS_SIM).result()
    counts = result.get_counts()
    est = counts_to_expected_phase(counts, t)
    err = error_vs_true(est)
    ideal_errors.append(err)
    print(f"t={t}: estimate={est:.6f}, error={err:.6f}")

# Theoretical error bound: 2^-t
theory_errors = [2 ** (-t) for t in t_range]

The error falls as $2^{-t}$, as expected, and is zero at $t = 4$ because $5/16$ is exact in 4 bits.

## Precision sweep on hardware

The same sweep now runs on the device set in the next cell, Rigetti Cepheus by default. The number of shots per point is `SHOTS_HW` in the Setup cell. With more precision qubits the results spread over more outcomes, so a stable estimate needs more shots, and more shots cost more.

In [ ]:
# Device-specific transpilation.
#
# `device.profile.basis_gates` is None on every device except IonQ, so a plain
# `basis_gates=(... or None)` silently transpiles for no device at all and every backend
# below produces identical rows. On IonQ it raises instead, because its native names
# (gpi, gpi2, si, vi) are not Qiskit-standard. Every gate in HW_BASIS below IS in
# IonQ's accepted set, so this should work there, but it is not yet verified on IonQ.
#
# So: name the gate set explicitly, and take the qubit layout from `device.coupling_map` on
# newer qBraid SDKs, or from the device topology qBraid publishes on SDK 0.12. Trapped-ion machines report no coupling map because they are fully
# connected - no routing, no SWAPs - and that contrast is part of the lesson.
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']


def coupling_for(device):
    """Which qubits are connected, or None if every qubit connects to every other."""
    cmap = getattr(device, 'coupling_map', None)             # available on newer qBraid SDKs
    if cmap:
        return [list(edge) for edge in cmap]
    topology = provider.client.get_device(device.id).topology or {}   # qBraid SDK 0.12
    if 'fully' in topology.get('type', '') or not ('rows' in topology or 'rowSpans' in topology):
        return None
    spans = topology.get('rowSpans') or [[0, topology['cols'] - 1]] * topology['rows']
    index = {}
    for r, (c0, c1) in enumerate(spans):                      # number the grid sites row by row
        for c in range(c0, c1 + 1):
            index[(r, c)] = len(index)
    edges = []
    for (r, c), i in index.items():                           # connect each site to its right and lower neighbour
        for nb in ((r, c + 1), (r + 1, c)):
            if nb in index:
                edges += [[i, index[nb]], [index[nb], i]]
    return edges


def transpile_for(device, circuit, optimization_level=2):
    """Transpile for a specific device: its qubit layout and an explicit gate set."""
    return transpile(
        circuit,
        coupling_map=coupling_for(device),
        basis_gates=HW_BASIS,
        optimization_level=optimization_level,
    )


provider = QbraidProvider()

# Devices are named by qBraid QRN. The README lists devices, prices and availability.
DEVICE_ID = 'rigetti:rigetti:qpu:cepheus-1-108q'
device = provider.get_device(DEVICE_ID)


hardware_errors = []
hardware_stats = []
for t in t_range:
    qc = build_qpe_circuit(t=t)

    # Report circuit depth and gate count before submitting
    transpiled = transpile_for(device, qc)
    depth = transpiled.depth()
    n_gates = sum(transpiled.count_ops().values())
    n_2q = sum(v for k, v in transpiled.count_ops().items() if k in ['cx', 'cz', 'iswap', 'ecr'])
    hardware_stats.append({'t': t, 'depth': depth, 'gates': n_gates, '2Q gates': n_2q})

    print(f"t={t}: depth={depth}, gates={n_gates}, 2Q gates={n_2q}. Submitting...")
    job = device.run(qc, shots=SHOTS_HW, tags=QUEST_JOB_TAGS)
    result = job.result()
    counts = result.data.get_counts()
    est = counts_to_expected_phase(counts, t)
    err = error_vs_true(est)
    hardware_errors.append(err)
    print(f"  -> estimate={est:.4f}, error={err:.4f}")

pd.DataFrame(hardware_stats).set_index('t')

## Ideal versus hardware

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.semilogy(t_range, theory_errors, 'k--', label='Theoretical bound ($2^{-t}$)', linewidth=2, alpha=0.5)
ax.semilogy(t_range, [max(e, 1e-4) for e in ideal_errors], 'o-', color='#2d7a4f',
            label='Ideal simulator', markersize=12, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)
ax.semilogy(t_range, hardware_errors, 's-', color='#c63792',
            label=f'Real hardware ({DEVICE_ID})', markersize=12, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('Precision qubits (t)', fontsize=12)
ax.set_ylabel('Phase estimation error', fontsize=12)
ax.set_title('QPE precision sweep: ideal vs. real hardware', fontsize=13, pad=15)
ax.set_xticks(t_range)
ax.grid(alpha=0.3, which='both')
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

On the simulator the error keeps falling. On hardware it usually falls at first and then levels off or rises. Each device has a best number of precision qubits, and going beyond it makes the estimate worse.

A noise-free simulator cannot show this. On current hardware the precision of QPE is limited by noise from circuit depth more than by the number of shots, because each extra digit doubles the length of the deepest part of the circuit.

## Optional, advanced: Zero-noise extrapolation

Error mitigation can recover some of the lost accuracy. Zero-noise extrapolation (ZNE) is one of the simplest methods.

The idea is to run the same circuit at several noise scale factors $\lambda = 1, 3, 5$, where $\lambda = 1$ is the device's normal noise and larger values add noise on purpose. Fitting the results against $\lambda$ and extending the fit to $\lambda = 0$ estimates the noise-free answer.

Here the noise is scaled by **global folding**. The whole circuit $U$ is replaced by $U\,(U^\dagger U)^n$. Since $U^\dagger U$ is the identity, the ideal result is unchanged, but the circuit is $2n+1$ times as long and collects roughly $2n+1$ times the noise.

We fix $t = 4$ and use scale factors 1, 3 and 5.

In [ ]:
from qiskit import transpile

def fold_gates_global(qc, scale_factor):
    """
    Global folding: replace circuit U with (U^-1 U)^n U for scale factor 2n+1.
    Requires scale_factor to be odd (1, 3, 5, ...).
    """
    if scale_factor == 1:
        return qc
    if scale_factor % 2 == 0:
        raise ValueError('Scale factor must be odd for global folding')

    n = (scale_factor - 1) // 2

    # Build the inverse of the measured circuit
    qc_measure = None
    qc_pure = qc.copy()
    # Remove measurements for folding
    qc_pure.remove_final_measurements(inplace=True)

    qc_inv = qc_pure.inverse()

    folded = qc_pure.copy()
    for _ in range(n):
        folded.compose(qc_inv, inplace=True)
        folded.compose(qc_pure, inplace=True)

    # Re-add measurements
    folded.measure_all()
    return folded


# Test that folding preserves the ideal result
qc_base = build_qpe_circuit(t=4)
qc_folded_3 = fold_gates_global(qc_base, scale_factor=3)
qc_folded_5 = fold_gates_global(qc_base, scale_factor=5)

print(f"Scale factor 1: depth={qc_base.depth()}, gates={sum(qc_base.count_ops().values())}")
print(f"Scale factor 3: depth={qc_folded_3.depth()}, gates={sum(qc_folded_3.count_ops().values())}")
print(f"Scale factor 5: depth={qc_folded_5.depth()}, gates={sum(qc_folded_5.count_ops().values())}")

# Verify on simulator that the folded circuits give the same answer as unfolded
for lam, qc in [(1, qc_base), (3, qc_folded_3), (5, qc_folded_5)]:
    result = sim.run(transpile(qc, sim), shots=SHOTS_SIM).result()
    counts_sim = result.get_counts()
    # For folded circuits qiskit may use measure_all which adds ancilla measurements
    # Filter to the first 4 bits
    counts_precision = {}
    for bs, ct in counts_sim.items():
        key = bs[-4:] if len(bs) > 4 else bs
        counts_precision[key] = counts_precision.get(key, 0) + ct
    est = counts_to_expected_phase(counts_precision, t=4)
    print(f"Scale {lam} on ideal simulator: estimate={est:.6f} (should be {TRUE_PHI})")

Folding leaves the ideal answer unchanged, as it should. Next, each folded circuit runs on hardware.

In [ ]:
scale_factors = [1, 3, 5]

zne_data = []
for lam in scale_factors:
    qc = fold_gates_global(build_qpe_circuit(t=4), scale_factor=lam)
    print(f"Running scale factor {lam} on hardware (this takes a few minutes)...")
    job = device.run(qc, shots=SHOTS_ZNE, tags=QUEST_JOB_TAGS)
    result = job.result()
    counts = result.data.get_counts()

    # Filter to first 4 bits if necessary
    counts_p = {}
    for bs, ct in counts.items():
        key = bs[-4:] if len(bs) > 4 else bs
        counts_p[key] = counts_p.get(key, 0) + ct

    est = counts_to_expected_phase(counts_p, t=4)
    err = error_vs_true(est)
    zne_data.append({'scale_factor': lam, 'estimate': est, 'error': err})
    print(f"  Scale factor {lam}: estimate={est:.4f}, error={err:.4f}")

zne_df = pd.DataFrame(zne_data)
zne_df

## Optional, advanced: Extrapolate to zero noise

A straight line through the points $(\lambda, \text{estimate})$, extended to $\lambda = 0$, gives the mitigated estimate.

In [ ]:
# Linear extrapolation
lambdas = np.array([d['scale_factor'] for d in zne_data])
estimates = np.array([d['estimate'] for d in zne_data])

# Fit y = a + b*lambda, extrapolate to lambda=0 means y_mitigated = a
p_linear = np.polyfit(lambdas, estimates, deg=1)
mitigated_linear = p_linear[1]  # intercept

# Richardson extrapolation (quadratic fit through same points)
p_richardson = np.polyfit(lambdas, estimates, deg=2)
mitigated_richardson = np.polyval(p_richardson, 0)

print(f"Raw estimate (scale=1):     {estimates[0]:.4f}  (error: {error_vs_true(estimates[0]):.4f})")
print(f"Linear ZNE:                 {mitigated_linear:.4f}  (error: {error_vs_true(mitigated_linear):.4f})")
print(f"Richardson ZNE:             {mitigated_richardson:.4f}  (error: {error_vs_true(mitigated_richardson):.4f})")
print(f"True value:                 {TRUE_PHI:.4f}")

In [ ]:
# Visualize the extrapolation
fig, ax = plt.subplots(figsize=(9, 6))

# Data points
ax.plot(lambdas, estimates, 'o', color='#c63792', markersize=14, label='Hardware measurements',
        markeredgecolor='white', markeredgewidth=1.5, zorder=3)

# Extrapolation lines
lam_range = np.linspace(-0.5, 5.5, 100)
ax.plot(lam_range, np.polyval(p_linear, lam_range), '--', color='#2d7a4f',
        label=f'Linear fit -> {mitigated_linear:.4f}', linewidth=2)
ax.plot(lam_range, np.polyval(p_richardson, lam_range), ':', color='#1a5285',
        label=f'Richardson fit -> {mitigated_richardson:.4f}', linewidth=2)

# True value
ax.axhline(TRUE_PHI, color='k', linewidth=1.5, alpha=0.4, label=f'True phi = {TRUE_PHI}')
ax.axvline(0, color='k', linewidth=0.5, alpha=0.3)

ax.set_xlabel(r'Noise scale factor $\lambda$', fontsize=12)
ax.set_ylabel('Phase estimate', fontsize=12)
ax.set_title('Zero-noise extrapolation for QPE at t=4', fontsize=13, pad=15)
ax.set_xlim(-0.5, 5.5)
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

ZNE often recovers part of the ideal accuracy. The cost is extra runs: here three circuits instead of one, and the folded circuits are longer.

ZNE assumes the result changes smoothly with the noise level. That holds for many measurements. It fails when the circuit is so noisy that every scale factor gives a nearly random result, because then there is no trend to extrapolate.

## When to use QPE, and when to use something else

On today's hardware QPE is limited to a few precision qubits.

QPE is a good fit when:

- you need one specific eigenvalue to a known precision;
- controlled-$U$ can be built from a small number of native gates;
- the device, or error mitigation, can support the circuit depth.

A variational method such as VQE is often more practical when:

- you need the ground state or a few low-lying states;
- the Hamiltonian is a sum of Pauli terms, as in Ising models and molecules;
- a shorter circuit matters more than a convergence guarantee;
- you can afford an iterative classical optimization.

QPE returns a specific eigenvalue to a chosen precision. VQE returns an approximation to a specific eigenstate. Today VQE is usually preferred for chemistry and materials problems because its circuits are shorter. QPE becomes more practical as hardware and error correction improve.

## Going further

- **Use a phase that is not an exact binary fraction.** Set `TRUE_PHI = 1/3` in the Setup cell and repeat the sweep. The estimates no longer land on a single outcome, and the error pattern is less regular.
- **Try other mitigation methods.** The qBraid Error-Mitigation series (`tutorials/Error-Mitigation/` in this repository) covers readout mitigation and ZNE in more detail.
- **Compare with a variational method.** The [VQE notebook](../chemistry_and_physics/advanced_01_vqe_h2_ground_state.ipynb) estimates the ground-state energy of $H_2$ on hardware.